# Analise Exploratoria - Household Electricity Survey (EV0702)

Notebook para analise dos dados extraidos pelo pipeline Medallion (Bronze -> Silver -> Gold).

Os dados sao lidos diretamente do bucket **gold** do MinIO.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Tenta conectar ao MinIO; se nao disponivel, le do filesystem local
GOLD_DIR = None
_minio_client = None

try:
    from io import BytesIO
    from minio import Minio
    _minio_client = Minio(
        "localhost:9000",
        access_key="minioadmin",
        secret_key="minioadmin",
        secure=False,
    )
    _minio_client.list_buckets()
    print("Conectado ao MinIO.")
except Exception:
    GOLD_DIR = os.path.join(os.path.dirname(os.getcwd()), "data", "gold", "pipeline")
    if not os.path.exists(GOLD_DIR):
        GOLD_DIR = os.path.join(os.getcwd(), "data", "gold", "pipeline")
    print(f"MinIO indisponivel. Lendo CSVs de: {GOLD_DIR}")

def read_gold_csv(name: str) -> pd.DataFrame:
    if _minio_client:
        response = _minio_client.get_object("gold", f"pipeline/{name}.csv")
        df = pd.read_csv(BytesIO(response.read()))
        response.close()
        response.release_conn()
        return df
    else:
        return pd.read_csv(os.path.join(GOLD_DIR, f"{name}.csv"))

## 1. Consumo Medio por Tipo de Eletrodomestico

In [2]:
df_appliance = read_gold_csv("appliance_avg_consumption")
print(f"Shape: {df_appliance.shape}")
df_appliance

MaxRetryError: HTTPConnectionPool(host='localhost', port=9000): Max retries exceeded with url: /gold?location= (Caused by ProtocolError('Connection aborted.', BadStatusLine('ÿ\x00\x00\x00\x00\x00\x00\x00\x01\x7f:\x01000\r\n')))

In [ ]:
df_appliance.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
df_sorted = df_appliance.sort_values("avg_annual_consumption_kwh", ascending=True)
colors = sns.color_palette("YlOrRd", len(df_sorted))
ax.barh(df_sorted["appliance_type"], df_sorted["avg_annual_consumption_kwh"], color=colors)
ax.set_xlabel("Consumo Anual Medio (kWh)")
ax.set_title("Consumo Medio Anual por Tipo de Eletrodomestico")
for i, v in enumerate(df_sorted["avg_annual_consumption_kwh"]):
    ax.text(v + 10, i, f"{v:.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Economia Potencial por Eletrodomestico

In [ ]:
df_savings = read_gold_csv("potential_savings")
print(f"Shape: {df_savings.shape}")
df_savings

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
df_sav_sorted = df_savings.sort_values("avg_annual_savings_kwh", ascending=True)
colors = sns.color_palette("Greens", len(df_sav_sorted))
ax.barh(df_sav_sorted["appliance_type"], df_sav_sorted["avg_annual_savings_kwh"], color=colors)
ax.set_xlabel("Economia Anual Media (kWh)")
ax.set_title("Economia Potencial por Tipo de Eletrodomestico")
for i, v in enumerate(df_sav_sorted["avg_annual_savings_kwh"]):
    ax.text(v + 3, i, f"{v:.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Consumo vs Economia - Comparativo

In [ ]:
df_merged = pd.merge(
    df_appliance, df_savings,
    on="appliance_type", how="inner"
)
df_merged["savings_pct"] = (
    df_merged["avg_annual_savings_kwh"] / df_merged["avg_annual_consumption_kwh"] * 100
).round(1)
df_merged.sort_values("savings_pct", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
x = range(len(df_merged))
width = 0.35
ax.bar([i - width/2 for i in x], df_merged["avg_annual_consumption_kwh"], width, label="Consumo (kWh)", color="#e74c3c")
ax.bar([i + width/2 for i in x], df_merged["avg_annual_savings_kwh"], width, label="Economia Potencial (kWh)", color="#2ecc71")
ax.set_xticks(x)
ax.set_xticklabels(df_merged["appliance_type"], rotation=45, ha="right")
ax.set_ylabel("kWh/ano")
ax.set_title("Consumo vs Economia Potencial por Eletrodomestico")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Consumo Anual por Tipo de Residencia

In [ ]:
df_household = read_gold_csv("annual_consumption_by_household")
print(f"Shape: {df_household.shape}")
df_household

In [ ]:
df_household.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

metrics = [
    ("kwh_per_household", "kWh por Residencia"),
    ("kwh_per_m2", "kWh por m2"),
    ("kwh_per_person", "kWh por Pessoa"),
]

for ax, (col, title) in zip(axes, metrics):
    data = df_household.dropna(subset=[col]).sort_values(col, ascending=True)
    ax.barh(data["household_type"], data[col], color=sns.color_palette("Blues", len(data)))
    ax.set_title(title)
    ax.set_xlabel("kWh/ano")

plt.suptitle("Consumo Anual por Tipo de Residencia", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Breakdown de Consumo por Categoria

In [ ]:
df_breakdown = read_gold_csv("consumption_breakdown_by_category")
print(f"Shape: {df_breakdown.shape}")
df_breakdown.head(20)

In [ ]:
if "without_heating_all_days" in df_breakdown.columns:
    pivot = df_breakdown.pivot_table(
        index="category",
        columns="household_type",
        values="without_heating_all_days",
        aggfunc="first"
    )
    fig, ax = plt.subplots(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Distribuicao de Consumo (%) por Categoria - Sem Aquecimento Eletrico (Todos os Dias)")
    ax.set_ylabel("Categoria")
    ax.set_xlabel("Tipo de Residencia")
    plt.tight_layout()
    plt.show()

## 6. Consumo Detalhado por Tipo de Casa e Aquecimento

In [ ]:
df_detailed = read_gold_csv("detailed_consumption_by_house_type")
print(f"Shape: {df_detailed.shape}")
df_detailed.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
for category in df_detailed["heating_category"].unique():
    subset = df_detailed[df_detailed["heating_category"] == category]
    subset = subset.dropna(subset=["kwh_per_household"])
    ax.barh(
        subset["house_type"] + " (" + category + ")",
        subset["kwh_per_household"],
        label=category,
        alpha=0.8,
    )
ax.set_xlabel("kWh por Residencia/ano")
ax.set_title("Consumo por Tipo de Casa e Categoria de Aquecimento")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Resumo Estatistico Geral

In [ ]:
print("=" * 60)
print("RESUMO DOS DADOS GOLD")
print("=" * 60)

datasets = {
    "Consumo por Eletrodomestico": df_appliance,
    "Economia Potencial": df_savings,
    "Consumo por Tipo de Residencia": df_household,
    "Breakdown por Categoria": df_breakdown,
    "Consumo Detalhado": df_detailed,
}

for name, df in datasets.items():
    print(f"\n--- {name} ---")
    print(f"  Linhas: {len(df)} | Colunas: {len(df.columns)}")
    print(f"  Colunas: {list(df.columns)}")
    numeric_cols = df.select_dtypes(include=["number"]).columns
    if len(numeric_cols) > 0:
        print(f"  Colunas numericas: {list(numeric_cols)}")
        for col in numeric_cols:
            vals = df[col].dropna()
            if len(vals) > 0:
                print(f"    {col}: min={vals.min():.1f}, max={vals.max():.1f}, mean={vals.mean():.1f}, std={vals.std():.1f}")